# tom — full training pipeline on Colab (T4 GPU)

Runs the four-stage RoBERTa pipeline (DAPT → search → final training → ensemble) end to end.
**Runtime → Change runtime type → T4 GPU** before running. Total ~2–4 GPU-hours.

Progress is synced to your Google Drive after every stage, and every stage resumes
from partial results — if Colab disconnects, just *Run all* again.

In [ ]:
# 1 · GPU check
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
# 2 · Mount Drive (progress backup lives here)
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/tom_runs'
!mkdir -p {BACKUP}

In [ ]:
# 3 · Clone the repo and restore any previous progress
%cd /content
!test -d COMP9444_26T2_FOMC_Analysis || git clone --depth 1 https://github.com/StrawHatSWE/COMP9444_26T2_FOMC_Analysis
%cd COMP9444_26T2_FOMC_Analysis/tom
!mkdir -p runs && cp -rn {BACKUP}/. runs/ 2>/dev/null; echo "restored:" && ls runs || true

def sync():
    get_ipython().system(f'cp -r runs/. {BACKUP}/')
    print('synced runs/ -> Drive')

In [ ]:
# 4 · Dependencies (torch is preinstalled on Colab)
!pip install -q transformers datasets openpyxl "accelerate>=0.26" 

## Stage 1 — DAPT (~40–60 min)
Masked-language-model continuation on the 47k-sentence Fed corpus (test-filtered at build
time). Output: an adapted encoder at `runs/dapt/`.

In [ ]:
!python dapt.py --corpus fed_corpus.txt
sync()

## Stage 2 — Hyperparameter search (~30–45 min)
9 candidate recipes, scored on seed-5768 validation only. Resumes per candidate.
Output: `runs/search/selected_config.json` (the frozen recipe) + the full ranking CSV.

In [ ]:
!python search.py --model runs/dapt
sync()
!cat runs/search/selected_config.json

## Stage 3 — Final training (~20–60 min depending on the winning recipe)
Frozen recipe × 3 seeds × 3 restarts = 9 models, each evaluated once on its own raw test file.
Resumes per model.

In [ ]:
!python train_final.py --restarts 3
sync()

## Stage 4 — Ensemble (~2 min)
Within-seed softmax averaging of the 3 restarts; the reported number is mean ± std of the
three per-seed ensembles.

In [ ]:
!python ensemble.py --device cuda
sync()

In [ ]:
# 5 · Results summary
import json
from pathlib import Path
single = json.loads(Path('runs/final/results.json').read_text())
ens = json.loads(Path('runs/final/results_ensemble.json').read_text())
print(f"single-model  F1: {single['single_model_mean_f1']:.4f} +/- {single['single_model_std_f1']:.4f}")
print(f"ensemble      F1: {ens['mean_f1']:.4f} +/- {ens['std_f1']:.4f}   acc {ens['mean_accuracy']:.4f}")
print("
baselines: tuned FinBERT 0.629 +/- 0.011 | published ceiling (RoBERTa-large) 0.71-0.74")
print("
per seed:", json.dumps(ens['per_seed'], indent=2))

## Optional — RoBERTa-large (~1.5–2 h extra)
Only after the base run succeeds. Large is fine-tuning-unstable: use lr 1e-5; if a seed
collapses to all-neutral, delete its `metrics-*.json` and re-run the cell.

```
!python dapt.py --corpus fed_corpus.txt --model roberta-large --output-dir runs/dapt-large --batch-size 8
!python search.py --model runs/dapt-large --output-dir runs/search-large --candidates large_candidates.json
!python train_final.py --config runs/search-large/selected_config.json --restarts 3 --output-dir runs/final-large
!python ensemble.py --models-dir runs/final-large --device cuda
```
(`large_candidates.json`: a trimmed 4-candidate list around lr 1e-5 — ask Claude to generate it,
or copy CANDIDATES from search.py and drop the 3e-5 rows.)